# Phase 7: Evaluation and Transaction-Cost Extension

This phase summarises the project and adds the proposal's primary extension from section 3.7: subtracting proportional transaction costs from adopter profit and asking whether the rule's modest statistical edge survives economically. The simulator dynamics are unchanged; costs enter only the P&L accounting, so a single set of paired-shock seed runs supports a sweep over many cost levels.

Reference: section 4.2 of the proposal for the diagnostic set, section 3.7 for the transaction-cost extension.

## Summary report

### Headline framing

What the simulator actually shows is a **dual-channel result** that pulls in opposite directions along different evaluation targets on the same path. The right one-line summary is:

> Adoption of the shared AR forecast raises realised-return predictive performance through the self-fulfilment channel, while reducing the independent demand-adjusted signal that the forecast was originally designed to exploit. Economic value does not visibly erode in the cost-free baseline.

Wherever this notebook says "erosion" without further qualification it means the demand-adjusted channel only. Along realised returns and along economic P&L the same forecast moves in the *opposite* direction in the cost-free baseline, and transaction costs (this phase) shift profit levels but do not reverse the adoption-amplification pattern in the regimes tested.

### Core findings (phases 1-6)

1. **Two-channel R^2 result (phase 4).** The same rolling AR forecast both *gains* predictive power against realised returns and *loses* it against the demand-adjusted return `r_{t+1} - mu D_t = phi r_t + sigma eps`, as adoption rises. The two views correspond to the realised-return endpoint (section 4.1.1 of the proposal) and the demand-adjusted-erosion endpoint (section 4.1.2). Both are real properties of the same forecast on the same path; they answer different questions, not the same question with conflicting answers. Note that the demand-adjusted return is a counterfactual that strips out contemporaneous price impact, not the full regression residual.
2. **Effective phi rises rather than falls.** In the implemented return law `r_{t+1} = phi r_t + mu D_t + shock`, adopter buying on a positive AR forecast reinforces the lagged-return component through `mu D_t`. Rolling lag-1 autocorrelation grows from input phi (0.25) to ~0.45 at full adoption. The underlying demand-adjusted phi is fixed by construction; the rolling fit picks up the amplified empirical phi and over-predicts the demand-adjusted process, which is the mechanical reason the demand-adjusted R^2 falls.
3. **Relative critical adoption shares (phase 6).** Because the cost-free baseline never produces an absolute zero crossing in demand-adjusted R^2, phase 6 measures **relative** thresholds: `A*_{R2,da,rel}` is the smallest A at which demand-adjusted R^2 has fallen to half its baseline, and `A*_{phi,rel}` is the smallest A at which effective phi has grown to 1.5x its baseline. These are *not* the proposal's section-4.3 zero-crossing `A*`; they are operational analogues that are well-defined in the cost-free regime. `A*_{R2,realised}` is "not reached" because realised R^2 rises with adoption rather than falls. Across the (mu, phi, w) grid larger mu generally pulls `A*_{R2,da,rel}` to the left (stronger feedback bites earlier), but the phi gradient is not mechanically monotone: stronger underlying predictability raises the baseline R^2, which interacts with the relative threshold in a way that is best read off the heatmap rather than summarised as a one-dimensional trend. The effective-phi heatmap `A*_{phi,rel}` is the cleaner read of the adoption-feedback channel because it does not depend on the AR fit quality.

### Robustness checks (already in the repo)

- **100-seed Monte Carlo bands (phase 4).** Both R^2 channels and the effective-phi trajectory hold in expectation with tight +/-1 SD bands across paired-shock seeds. Not a single-seed coincidence.
- **Stochastic vs performance-based switching (phase 5).** Endogenous CE-based switching reaches the same terminal demand-adjusted-R^2 erosion and effective-phi amplification as exogenous diffusion; only the path through adoption space differs. The dual-channel pattern does not depend on the specific switching rule.
- **AR order sensitivity (phase 6).** The demand-adjusted-channel erosion and effective-phi amplification gradients survive `p = 1, 2, 5, 10`. The delta-from-AR(1) heatmap is mostly near zero, which is the cleanest single robustness check.
- **Hit-rate filtered grid (phase 6).** Relative-threshold values are reported only when at least 50% of seeds cross the threshold, with the actual hit rate annotated on the heatmap when filtered out. No noisy single seeds set a cell.
- **Paired-shock invariant (phase 4 simulator fix).** Adoption parameters never advance the market RNG, so cross-regime comparisons sit on the same shock stream.

### Boundary conditions

- **Weak-signal regime.** At phi < 0.10 combined with short forecast windows (w = 100), the rolling fit has essentially no signal to lose. The relative R^2 threshold is meaningless (baseline is at or below zero), and the hit-rate filter correctly marks those cells NaN.
- **Cost-free baseline does not show economic erosion.** Mean adopter profit *grows* with adoption when adopter trades coordinate (q^A is capped, demand pushes price in the forecast direction, adopters earn the amplified move). A simple economic-erosion claim ("the rule stops paying off as adoption rises") is *not* supported by this model in the cost-free baseline along either the realised-return endpoint or the economic endpoint.
- **Endogenous switching reinforces, does not dampen.** With no costs, the CE score stays positive at full adoption, so endogenous switching has no reason to back off. A negative-feedback CE only emerges in the presence of costs, which is what this phase tests.

### Phase 7 question

At what proportional cost level c does mean adopter net profit cross zero, and does the cost level required to make adoption unprofitable shrink with adoption itself? The proposal's economic endpoint compares against the null benchmark, not against zero. The null-relative net profit is therefore reported as the proposal's primary economic measure, with absolute net profit reported as a secondary diagnostic.

In [ ]:
# Common run parameters (mirror phase 4's well-behaved baseline).
N = 200
T = 8000
mu = 0.05
phi = 0.25
sigma_news = 0.01
sigma_q = 1.0
forecast_window = 250
forecast_p = 1
risk_scale = 0.001
q_cap = 0.05
eval_window = 1000
adoption_start_t = forecast_window + eval_window
adoption_pi = 3e-4   # slow stochastic diffusion: A ramps from 0 to ~0.87 over the run

num_seeds_phase7 = 50
base_seed = 4000

# Cost levels (proportional cost per unit |q^A| traded each period).
# c = 0 reproduces the phase 4 baseline. The absolute crossing zone
# (negative profit at low A, positive at high A) sits around c = 0.003
# to 0.0045 in this regime, so the grid resolves that range while
# keeping a few extreme values as sanity points.
c_grid = [0.0, 0.001, 0.002, 0.003, 0.004, 0.005, 0.0075, 0.01]

# Window for binning rolling net profit by adoption share.
n_bins = 30

fig_dir = "../results/figures"
data_dir = "../results/data"

In [ ]:
import sys
sys.path.insert(0, "../src")

import numpy as np
import matplotlib.pyplot as plt
from joblib import Parallel, delayed

from reflexive_market import simulate

In [ ]:
def run_seed(seed_s):
    rng_s = np.random.default_rng(seed_s)
    out = simulate.run(
        T=T, N=N, mu=mu, phi=phi,
        sigma_news=sigma_news, sigma_q=sigma_q,
        rng=rng_s,
        forecast_window=forecast_window, forecast_p=forecast_p,
        risk_scale=risk_scale, q_cap=q_cap,
        adoption_pi=adoption_pi, adoption_delta=0.0,
        adoption_start_t=adoption_start_t,
    )
    # Recompute q_adv (the per-period advanced order size) from the
    # forecasts series. The simulator does not export it directly, but
    # it is fully determined by forecast, risk_scale, and q_cap.
    q_adv = np.where(
        np.isfinite(out["forecasts"]),
        np.clip(out["forecasts"] / risk_scale, -q_cap, q_cap),
        0.0,
    )
    return {
        "adoption_share": out["adoption_share"],
        "returns": out["returns"],
        "q_adv": q_adv,
        "abs_q_adv": np.abs(q_adv),
        "gross_profit": q_adv * out["returns"],
        # Realised null-rule gross P&L per period (population-average over
        # null traders): mean(null_orders) * r_{t+1}. Non-zero in expectation
        # because null orders contribute to D_t and so are correlated with
        # the contemporaneous return through mu D_t. The null benchmark in
        # the relative endpoint must include this term.
        "null_profit": out["null_profit"],
    }


seeds = [base_seed + s for s in range(num_seeds_phase7)]
raw = Parallel(n_jobs=-1, verbose=5)(delayed(run_seed)(s) for s in seeds)
print(f"Completed {len(raw)} seed runs.")

In [ ]:
# Pool (A_t, gross_profit_t, abs_q_adv_t, null_profit_t) across all seeds
# and time, then compute net profit (both absolute and null-relative) for
# each cost level and bin by adoption share.
warmup = adoption_start_t
A_pool = np.concatenate([r["adoption_share"][warmup:] for r in raw])
gross_pool = np.concatenate([r["gross_profit"][warmup:] for r in raw])
abs_q_pool = np.concatenate([r["abs_q_adv"][warmup:] for r in raw])
null_pool = np.concatenate([r["null_profit"][warmup:] for r in raw])
print(f"Pooled observations: {A_pool.size:,}")

bin_edges = np.linspace(0.0, 1.0, n_bins + 1)
bin_centers = 0.5 * (bin_edges[:-1] + bin_edges[1:])


def bin_stats(x, y, edges):
    inds = np.digitize(x, edges) - 1
    nb = len(edges) - 1
    inds = np.clip(inds, 0, nb - 1)
    means = np.full(nb, np.nan)
    stds = np.full(nb, np.nan)
    counts = np.zeros(nb, dtype=int)
    for i in range(nb):
        m = inds == i
        c = int(m.sum())
        counts[i] = c
        if c > 1:
            means[i] = y[m].mean()
            stds[i] = y[m].std()
    return means, stds, counts


# Null-rule benchmark per representative null trader, with cost on |q^(0)|.
# A null trader draws q^(0) ~ N(0, sigma_q^2). Per-period gross P&L per
# representative trader is q^(0) * r, which we approximate by the simulator's
# population-average null_profit = mean(null_orders) * r (this is non-zero
# when adopter demand correlates returns with null orders through mu D_t,
# and is small but not negligible at low costs). Per-trader cost is c times
# E[|q^(0)|] = c * sqrt(2/pi) * sigma_q. So:
#   net_null[t] = null_profit[t] - c * sqrt(2/pi) * sigma_q
# The relative endpoint subtracts net_null from net_advanced.
null_mean_abs = float(np.sqrt(2.0 / np.pi) * sigma_q)

binned_abs = {}
binned_rel = {}
for c in c_grid:
    net_abs = gross_pool - c * abs_q_pool
    net_null = null_pool - c * null_mean_abs
    net_rel = net_abs - net_null
    binned_abs[c] = dict(zip(("means", "stds", "counts"), bin_stats(A_pool, net_abs, bin_edges)))
    binned_rel[c] = dict(zip(("means", "stds", "counts"), bin_stats(A_pool, net_rel, bin_edges)))

# Headline summary at low and high adoption.
low_mask = A_pool < 0.2
high_mask = A_pool > 0.6
print(
    f"\n{'c':>8} | "
    f"{'abs_low':>11} {'abs_high':>11} | "
    f"{'rel_low':>11} {'rel_high':>11} | "
    f"{'abs_flip':>9} {'rel_flip':>9}"
)
print("-" * 88)
summary_rows = []
for c in c_grid:
    net_abs = gross_pool - c * abs_q_pool
    net_null = null_pool - c * null_mean_abs
    net_rel = net_abs - net_null
    a_lo = float(net_abs[low_mask].mean()); a_hi = float(net_abs[high_mask].mean())
    r_lo = float(net_rel[low_mask].mean()); r_hi = float(net_rel[high_mask].mean())
    a_flip = "yes" if (a_lo > 0) != (a_hi > 0) else "no"
    r_flip = "yes" if (r_lo > 0) != (r_hi > 0) else "no"
    print(
        f"{c:>8.4f} | {a_lo:>+11.6f} {a_hi:>+11.6f} | "
        f"{r_lo:>+11.6f} {r_hi:>+11.6f} | "
        f"{a_flip:>9} {r_flip:>9}"
    )
    summary_rows.append((c, a_lo, a_hi, r_lo, r_hi))

## Mean rolling net profit vs adoption share, two views

The proposal's economic endpoint compares the adopter rule against the null benchmark, not against zero. Both views are useful and they live on the same scale.

**Absolute (left).** Per-period adopter net profit `q^A r_{t+1} - c |q^A|`, in dollar terms. Tests whether adopter is in the black on a stand-alone basis.

**Null-relative (right).** Adopter net profit minus the null benchmark `null_profit_t - c * sqrt(2/pi) * sigma_q`. The null benchmark uses the simulator's realised gross null P&L `mean(null_orders) * r_{t+1}` (non-zero per period because null orders enter aggregate demand and correlate with returns through `mu D_t`) plus per-trader cost on `|q^(0)|`. Tests whether adopter beats the null rule. Since null traders have `|q^(0)| ~ sigma_q` whereas adopters have `|q^A| <= q_cap`, null pays much heavier per-period cost; together with the small null-gross term this defines the proposal's intended economic endpoint.

For both panels, the line is the bin mean across pooled seeds and time, the band is +/-1 SD across the in-bin observations.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5), sharex=True)
colours = plt.cm.viridis(np.linspace(0.0, 0.9, len(c_grid)))
panels = [
    (axes[0], binned_abs, "Absolute net profit"),
    (axes[1], binned_rel, "Net profit relative to null benchmark"),
]
for ax, store, title in panels:
    for c, colour in zip(c_grid, colours):
        means = store[c]["means"]
        stds = store[c]["stds"]
        counts = store[c]["counts"]
        valid = counts > 10
        ax.plot(bin_centers[valid], means[valid], color=colour, linewidth=1.5, label=f"c = {c:g}")
        ax.fill_between(bin_centers[valid],
                         means[valid] - stds[valid],
                         means[valid] + stds[valid],
                         color=colour, alpha=0.12)
    ax.axhline(0.0, color="k", linewidth=0.5)
    ax.set_title(title)
    ax.set_xlabel("Adoption share")
    ax.set_ylabel("Mean net profit per period")
axes[0].legend(title="cost c", fontsize=8)
fig.suptitle(
    f"Adopter net profit vs adoption share (Monte Carlo, {num_seeds_phase7} seeds, mean +/- 1 SD)",
    y=1.02,
)
fig.tight_layout()
fig.savefig(f"{fig_dir}/phase_07_net_profit_vs_adoption.png", dpi=150, bbox_inches="tight")
plt.show()

## Per-cost critical adoption share, both views

For each c, the smallest A at which the binned mean crosses zero from below. Reported separately for the absolute and the null-relative versions, since the proposal's intended endpoint is the relative one.

In [ ]:
def first_zero_crossing(centers, means):
    """Smallest center at which the curve crosses zero from below.
    Returns NaN if the curve never goes from negative to positive.
    """
    valid = np.isfinite(means)
    if not valid.any():
        return np.nan
    cs = centers[valid]
    ms = means[valid]
    sign_change = (ms[:-1] < 0) & (ms[1:] >= 0)
    if not sign_change.any():
        return np.nan
    return float(cs[1:][sign_change][0])


# A*_{profit} comes in two flavours:
# - A*_{profit,abs}    : smallest A at which absolute adopter net profit crosses zero
# - A*_{profit,rel}    : smallest A at which null-relative adopter net profit crosses zero
# The proposal's primary economic endpoint is the null-relative one.
print(f"{'c':>8} | {'A*_profit_abs':>14} | {'A*_profit_rel':>14} | {'sign(rel) at A=0.5':>20}")
print("-" * 68)
a_star_profit_abs = np.full(len(c_grid), np.nan)
a_star_profit_rel = np.full(len(c_grid), np.nan)
for k, c in enumerate(c_grid):
    a_star_profit_abs[k] = first_zero_crossing(bin_centers, binned_abs[c]["means"])
    a_star_profit_rel[k] = first_zero_crossing(bin_centers, binned_rel[c]["means"])
    mid_idx = np.argmin(np.abs(bin_centers - 0.5))
    rel_mid = binned_rel[c]["means"][mid_idx]
    sign_str = "+" if rel_mid > 0 else "-" if rel_mid < 0 else "0"
    a_str = "-" if not np.isfinite(a_star_profit_abs[k]) else f"{a_star_profit_abs[k]:.3f}"
    r_str = "-" if not np.isfinite(a_star_profit_rel[k]) else f"{a_star_profit_rel[k]:.3f}"
    print(f"{c:>8.4f} | {a_str:>14} | {r_str:>14} | {sign_str:>20}")

In [ ]:
fig, ax = plt.subplots(figsize=(7, 4))
ax.plot(c_grid, a_star_profit_abs, marker="o", color="C3", label="A*_{profit,abs} (vs zero)")
ax.plot(c_grid, a_star_profit_rel, marker="s", color="C0", label="A*_{profit,rel} (vs null benchmark)")
ax.set_xlabel("Transaction cost c (per unit |q^A|)")
ax.set_ylabel("A*_{profit}")
ax.set_title("A*_{profit}: target-specific economic threshold per cost level")
ax.set_ylim(-0.02, 1.02)
ax.grid(True, alpha=0.3)
ax.legend()
fig.tight_layout()
fig.savefig(f"{fig_dir}/phase_07_a_star_profit_vs_cost.png", dpi=150)
plt.show()

## Save numeric summary

In [ ]:
summary = np.array(summary_rows)  # columns: c, abs_low, abs_high, rel_low, rel_high
binned_abs_means = np.array([binned_abs[c]["means"] for c in c_grid])
binned_abs_stds = np.array([binned_abs[c]["stds"] for c in c_grid])
binned_rel_means = np.array([binned_rel[c]["means"] for c in c_grid])
binned_rel_stds = np.array([binned_rel[c]["stds"] for c in c_grid])
binned_counts = np.array([binned_abs[c]["counts"] for c in c_grid])

np.savez(
    f"{data_dir}/phase_07_transaction_costs.npz",
    c_grid=np.array(c_grid),
    bin_centers=bin_centers,
    binned_abs_means=binned_abs_means,
    binned_abs_stds=binned_abs_stds,
    binned_rel_means=binned_rel_means,
    binned_rel_stds=binned_rel_stds,
    binned_counts=binned_counts,
    # Target-specific economic A* under both measurement conventions.
    a_star_profit_abs=a_star_profit_abs,
    a_star_profit_rel=a_star_profit_rel,
    null_mean_abs=np.array(null_mean_abs),
    summary=summary,
    phi=np.array(phi),
    mu=np.array(mu),
    risk_scale=np.array(risk_scale),
    q_cap=np.array(q_cap),
    forecast_window=np.array(forecast_window),
    eval_window=np.array(eval_window),
    adoption_pi=np.array(adoption_pi),
    T=np.array(T),
    N=np.array(N),
    num_seeds=np.array(num_seeds_phase7),
    base_seed=np.array(base_seed),
)

## Done when

- The summary report at the top distinguishes core findings, robustness checks, and boundary conditions in plain language under the dual-channel framing. (Done.)
- The transaction-cost extension produces two families of net-profit curves vs adoption share, one absolute and one null-relative, each spanning a range of cost levels.
- `A*_{profit,abs}` (absolute) is reached only in a narrow cost window (c around 0.003 to 0.004); outside that window the absolute curve either stays positive at all observed A (low c) or stays negative at all observed A (high c). `A*_{profit,rel}` (null-relative) is reached only at c = 0, around A ~ 0.38, where the null-relative curve starts slightly negative at low A and crosses to positive at moderate A. At every positive cost level the null-relative curve is uniformly positive because the null benchmark itself pays a heavy per-period cost on `|q^(0)| ~ sigma_q`.
- The result is reported under both endpoints. The proposal's primary economic endpoint (the null-relative one) shows that the rule beats the null benchmark across all positive cost levels tested, with one mild qualifier at c = 0; the absolute endpoint shows it survives standalone only at high enough adoption when costs are nontrivial.

### Connecting back to the proposal

The proposal's dual-channel framing predicts that adoption can move different evaluation targets in different directions on the same path, and this is what we observe. Demand-adjusted-channel predictive content erodes with adoption; realised-return performance and per-trade P&L both grow with adoption in the cost-free baseline. Transaction costs shift profit levels but do not reverse the adoption-amplification pattern: adopter mean profit still grows with adoption at every cost level tested, and the null benchmark loses more per period than adopters do because null traders pay cost on `|q^(0)| ~ sigma_q` while adopters pay cost on the much smaller `|q^A| <= q_cap`. The null-relative net profit is therefore mostly positive across the cost grid, with the single observed `A*_{profit,rel}` crossing happening at c = 0 around adoption ~ 0.38 (where the absence of any cost on the null benchmark makes its small per-period gross P&L just barely beat the adopter rule at very low adoption). The absolute net profit crosses zero in the c ~ 0.003 to 0.004 range and only at high enough adoption.

A simple "adoption erodes the strategy" claim would be inaccurate for this model. Under the dual-channel framing, the correct statement is: the independent demand-adjusted signal erodes; realised performance and economic value are amplified rather than eroded in the tested cost regimes; mechanisms that would also pull the realised/economic channels in the erosion direction (capacity constraints, heterogeneous adopter beliefs, fundamental price-impact penalties beyond the linear `mu D_t` term, or per-trader cost on `|q^A|` that scales with the number of adopters) are beyond this phase's scope. What this phase shows clearly is the relationship between cost and the adoption share at which the rule first pays off in absolute terms, and the contrast with the null-relative endpoint that the proposal explicitly favours; phase 4's dual-channel result and phase 6's threshold map remain the project's headline contributions.